In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [4]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [5]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [25]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores = eval_scores.sum()
eval_scores.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


torch.Size([100, 97])
tensor([-1.8483e+01,  1.4969e+01,  1.2362e+01,  5.7598e+01, -2.6601e+01,
         4.0696e+00, -4.0561e+00, -3.8956e+01, -1.7699e+01, -7.1053e+00,
        -1.6738e-01,  2.4903e-02,  6.1880e-01,  1.8604e-01,  1.3933e-01,
         3.1499e-01, -1.0980e-03,  6.9846e+00, -2.1928e-02,  2.6645e+01,
         3.6472e-03,  5.7017e-02,  1.2856e-01, -1.1743e+00,  3.1155e-01,
        -1.7285e-02,  5.8362e-02,  1.1587e-01,  1.5468e-01,  4.5747e-01,
         1.4429e+00,  9.5673e-01, -5.5046e-02,  0.0000e+00,  3.5000e+00,
         0.0000e+00,  7.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
        -1.7037e-02, -1.2579e-02,  0.0000e+00, -2.3230e-02,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00, -9.9343e-01,  0.0000e+00, -7.0000e+00, -7.5839e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  3.3072e+00, -2.7931e+00,
         0.0000e+00,  0.0000e

In [6]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [8]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

getting ref point
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(9,) (9,)
getting ref point
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(9,) (9,)


In [9]:
main_scorer(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.856154
Maximum Mean Discrepancy        0.003187
dtype: float64

In [10]:
detailed_scorer(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.933560
Min Objective Score: Knee Angle Error                                                                       186.338130
Min Objective Score: Hip Angle Error                                                                        822.850400
Min Objective Score: Arm Angle Error                                                                        861.650940
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.347920
Min Objective Score: Transverse Compliance                                                                  265.026400
Min Objective Score: Eccentric Compliance       